# 面试问题：怎样降低并评估 LLM 幻觉，引用和拒答机制如何设计？

**一句话回答**：先把“幻觉”拆成无证据主张、与证据矛盾、引用错位和不确定时不拒答。对需要事实的请求检索有版本/权限的证据；生成输出采用 claim→citation 结构；宿主验证引用存在、可见、支持对应 claim，覆盖不足或证据冲突时拒答/升级。评测同时分解 retrieval recall、claim groundedness、citation precision/coverage 与 abstention utility。

本 Notebook 在受控证据集上实现 claim/citation parser、支持度与数值矛盾检查、阈值校准、故障归因和 bootstrap。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib, json, math, re  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED110=11001; rng110=np.random.default_rng(SEED110)  # 计算并保存当前步骤的中间状态。
EVIDENCE110={"d1":{"tenant":"A","version":3,"text":"退款期限为7天。","time":30},"d2":{"tenant":"A","version":2,"text":"标准运费为10元。","time":20},"d3":{"tenant":"B","version":5,"text":"退款期限为30天。","time":40}}  # 计算并保存当前步骤的中间状态。
assert len(EVIDENCE110)==3  # 用受控断言验证关键不变量。
assert EVIDENCE110["d1"]["tenant"]=="A"  # 用受控断言验证关键不变量。
assert SEED110==11001  # 用受控断言验证关键不变量。

## 1. 把答案拆成原子 claim

一句话含多个事实时，一个引用可能只支持其中一半。输出最好直接使用结构化 claim 列表，每条包含文本、证据 ID 和置信/拒答状态。下面的简化句切分只用于受控中文示例，真实系统需处理指代、表格与隐含主张。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Claim110:  # 定义承载本节状态与行为的数据结构。
    text:str; citations:tuple  # 执行当前语句以推进本节示例。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.text or not isinstance(self.citations,tuple): raise ValueError("claim_contract")  # 按当前条件选择后续控制路径。
def split_claims110(answer):  # 定义本节可复用的核心函数。
    parts=[x.strip() for x in re.split(r"[。；]",answer) if x.strip()]; out=[]  # 计算并保存当前步骤的中间状态。
    for part in parts:  # 遍历输入元素以累积或检查结果。
        ids=tuple(re.findall(r"\[([^\]]+)\]",part)); clean=re.sub(r"\[[^\]]+\]","",part).strip(); out.append(Claim110(clean,ids))  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。
claims110=split_claims110("退款期限为7天[d1]。标准运费为10元[d2]。")  # 计算并保存当前步骤的中间状态。
assert len(claims110)==2 and claims110[0].citations==("d1",)  # 用受控断言验证关键不变量。
assert claims110[1].text=="标准运费为10元"  # 用受控断言验证关键不变量。
try: Claim110("",()); raise AssertionError("empty claim accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="claim_contract"  # 捕获预期异常并验证失败分支。

## 2. 引用首先要存在、版本正确且对当前主体可见

一个真实 ID 不代表可引用：跨租户、已删除、旧版本或来自召回之外的文档都必须拒绝。验证使用服务端证据集合，不能信任模型附带的 URL/标题。最终响应最好返回可追踪 doc/chunk/version，而非只给裸链接。

In [ ]:
def citation_access110(claim,tenant,retrieved_ids):  # 定义本节可复用的核心函数。
    errors=[]  # 计算并保存当前步骤的中间状态。
    for cid in claim.citations:  # 遍历输入元素以累积或检查结果。
        if cid not in EVIDENCE110: errors.append((cid,"missing"))  # 按当前条件选择后续控制路径。
        elif EVIDENCE110[cid]["tenant"]!=tenant: errors.append((cid,"forbidden"))  # 按当前条件选择后续控制路径。
        elif cid not in retrieved_ids: errors.append((cid,"not_retrieved"))  # 按当前条件选择后续控制路径。
    return errors  # 返回当前分支计算出的结果。
assert citation_access110(claims110[0],"A",{"d1","d2"})==[]  # 用受控断言验证关键不变量。
cross_claim110=Claim110("退款期限30天",("d3",)); assert citation_access110(cross_claim110,"A",{"d3"})==[("d3","forbidden")]  # 计算并保存当前步骤的中间状态。
assert citation_access110(Claim110("x",("missing",)),"A",set())==[("missing","missing")]  # 用受控断言验证关键不变量。

## 3. 支持度模型需要可验证 oracle

真实语义蕴含可用 NLI/LLM grader，但必须先在人工标注 claim-evidence 集上校准。这里实现字符/数字 token 覆盖作为透明 baseline：它只能发现明显缺词，不能理解否定、时间与逻辑，因此后续再加确定性数值矛盾规则。

In [ ]:
def tokens110(s): return set(re.findall(r"[A-Za-z0-9]+|[\u4e00-\u9fff]",s.lower()))  # 定义本节可复用的核心函数。
def lexical_support110(claim,evidence):  # 定义本节可复用的核心函数。
    ct=tokens110(claim); et=tokens110(evidence); return len(ct&et)/len(ct) if ct else 0.  # 计算并保存当前步骤的中间状态。
score_good110=lexical_support110("退款期限为7天",EVIDENCE110["d1"]["text"]); score_bad110=lexical_support110("退款期限为30天",EVIDENCE110["d1"]["text"])  # 计算并保存当前步骤的中间状态。
assert math.isclose(score_good110,1.)  # 用受控断言验证关键不变量。
assert 0<score_bad110<score_good110  # 用受控断言验证关键不变量。
assert lexical_support110("完全无关",EVIDENCE110["d1"]["text"])==0  # 用受控断言验证关键不变量。

## 4. 数字、日期、实体和否定做确定性矛盾检查

高风险字段先用 parser 比对：claim 中出现的金额/天数若不在证据中，标为 contradiction。多个证据互相冲突时按有效时间、权威级别和版本选择，若无法消解则展示冲突并拒绝下结论。

In [ ]:
def numbers110(s): return re.findall(r"\d+(?:\.\d+)?",s)  # 定义本节可复用的核心函数。
def numeric_relation110(claim,evidence):  # 定义本节可复用的核心函数。
    cn,en=set(numbers110(claim)),set(numbers110(evidence))  # 计算并保存当前步骤的中间状态。
    if not cn: return "not_applicable"  # 按当前条件选择后续控制路径。
    return "supported" if cn<=en else "contradiction"  # 返回当前分支计算出的结果。
assert numeric_relation110("退款期限为7天",EVIDENCE110["d1"]["text"])=="supported"  # 用受控断言验证关键不变量。
assert numeric_relation110("退款期限为30天",EVIDENCE110["d1"]["text"])=="contradiction"  # 用受控断言验证关键不变量。
latest110=max((d for d in EVIDENCE110.values() if d["tenant"]=="A"),key=lambda d:d["time"]); assert latest110["version"]==3  # 计算并保存当前步骤的中间状态。

## 5. Citation precision 与 claim coverage 分开报告

Citation precision 问“给出的引用有多少真正支持”；coverage 问“需要证据的 claim 有多少被支持”。删掉所有引用会让 precision 无定义却不能获得好分。还应报告 invalid/forbidden citation 和每条 claim 的证据数量。

In [ ]:
def citation_metrics110(claims,tenant,retrieved,threshold=.75):  # 定义本节可复用的核心函数。
    supported=0; valid_citations=0; supporting_citations=0  # 计算并保存当前步骤的中间状态。
    for c in claims:  # 遍历输入元素以累积或检查结果。
        claim_ok=False  # 计算并保存当前步骤的中间状态。
        for cid in c.citations:  # 遍历输入元素以累积或检查结果。
            if not citation_access110(c,tenant,retrieved) and cid in EVIDENCE110:  # 按当前条件选择后续控制路径。
                valid_citations+=1; ok=lexical_support110(c.text,EVIDENCE110[cid]["text"])>=threshold and numeric_relation110(c.text,EVIDENCE110[cid]["text"])!="contradiction"; supporting_citations+=int(ok); claim_ok|=ok  # 计算并保存当前步骤的中间状态。
        supported+=int(claim_ok)  # 计算并保存当前步骤的中间状态。
    return {"coverage":supported/len(claims) if claims else 1.,"precision":supporting_citations/valid_citations if valid_citations else 0.}  # 返回当前分支计算出的结果。
cm110=citation_metrics110(claims110,"A",{"d1","d2"})  # 计算并保存当前步骤的中间状态。
assert cm110=={"coverage":1.0,"precision":1.0}  # 用受控断言验证关键不变量。
bad_cm110=citation_metrics110([Claim110("退款期限30天",("d1",))],"A",{"d1"}); assert bad_cm110["coverage"]==0  # 计算并保存当前步骤的中间状态。
assert citation_metrics110([],"A",set())["coverage"]==1  # 用受控断言验证关键不变量。

## 6. 拒答阈值按业务效用校准

置信度可能由检索覆盖、支持度、冲突和模型不确定性组合。验证集上为“正确回答、错误回答、合理拒答”赋业务效用选阈值，高风险场景阈值更高。拒答率不是越低越好，要与错误放行率一起看。

In [ ]:
confidence110=np.array([.95,.8,.7,.55,.4,.2]); correct110=np.array([1,1,1,0,0,0])  # 计算并保存当前步骤的中间状态。
def utility110(threshold):  # 定义本节可复用的核心函数。
    answer=confidence110>=threshold; return float(np.sum(answer*np.where(correct110==1,1.,-3.)+(~answer)*np.where(correct110==1,-.25,.2)))  # 计算并保存当前步骤的中间状态。
thresholds110=np.unique(np.r_[0.,confidence110,1.]); utilities110=np.array([utility110(t) for t in thresholds110]); best_t110=float(thresholds110[np.argmax(utilities110)])  # 计算并保存当前步骤的中间状态。
assert .55<best_t110<=.7  # 用受控断言验证关键不变量。
assert utility110(best_t110)>utility110(0)  # 用受控断言验证关键不变量。
assert np.all((confidence110>=0)&(confidence110<=1))  # 用受控断言验证关键不变量。

## 7. 把端到端失败拆成检索、证据和生成

Gold evidence 未进入候选是 retrieval failure；候选有但被重排丢弃是 ranking failure；证据在上下文而 claim 不受支持是 generation/grounding failure；支持却引用错 ID 是 attribution failure。只有可归因，团队才知道该改哪个组件。

In [ ]:
def diagnose110(gold,retrieved,context,claim_supported,citation_correct):  # 定义本节可复用的核心函数。
    if gold not in retrieved: return "retrieval"  # 按当前条件选择后续控制路径。
    if gold not in context: return "ranking_or_budget"  # 按当前条件选择后续控制路径。
    if not claim_supported: return "generation_grounding"  # 按当前条件选择后续控制路径。
    if not citation_correct: return "attribution"  # 按当前条件选择后续控制路径。
    return "success"  # 返回当前分支计算出的结果。
assert diagnose110("d1",set(),set(),False,False)=="retrieval"  # 用受控断言验证关键不变量。
assert diagnose110("d1",{"d1"},set(),False,False)=="ranking_or_budget"  # 用受控断言验证关键不变量。
assert diagnose110("d1",{"d1"},{"d1"},True,False)=="attribution"  # 用受控断言验证关键不变量。

## 8. 评测与发布：对 claim 或文档族 bootstrap

黄金集包含 answerable/unanswerable、冲突证据、旧版本、跨租户和对抗引用。对独立问题或文档族 bootstrap，报告 coverage、precision、错误放行与合理拒答区间。上线监控引用点击/失效、拒答漂移和新失败回流。

In [ ]:
grounded110=np.array([1,1,0,1,1,1,0,1],float)  # 计算并保存当前步骤的中间状态。
def bootstrap_mean110(x,reps=2000,seed=110):  # 定义本节可复用的核心函数。
    rg=np.random.default_rng(seed); vals=np.array([x[rg.integers(0,len(x),len(x))].mean() for _ in range(reps)]); return float(x.mean()),tuple(np.quantile(vals,[.025,.975]))  # 计算并保存当前步骤的中间状态。
mean110,ci110=bootstrap_mean110(grounded110); manifest110={"schema":1,"evidence":"policy-v3","claim_parser":"v2","support":"lexical+noun-number-demo","abstain_threshold":best_t110,"tenant":"required"}; digest110=hashlib.sha256(json.dumps(manifest110,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert math.isclose(mean110,.75) and ci110[0]<=mean110<=ci110[1]  # 用受控断言验证关键不变量。
assert manifest110["tenant"]=="required" and 0<=manifest110["abstain_threshold"]<=1  # 用受控断言验证关键不变量。
assert len(digest110)==64  # 用受控断言验证关键不变量。

## 面试总结

回答应拆为：**原子 claim → 服务端证据/ACL/版本 → citation existence → 支持与矛盾 → precision/coverage → 效用校准拒答 → 检索/生成/归因故障树 → 分组评测与线上失效监控**。引用不是装饰，必须能让每个事实回到可访问、正确版本的证据。

延伸阅读：[NIST 对 Confabulation 的定义](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)、[FActScore](https://arxiv.org/abs/2305.14251)、[RAG](https://arxiv.org/abs/2005.11401)。